# CI/CD & Monitoring — Exercises

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 6/6
>
> Test your understanding of continuous integration, deployment patterns, drift detection, and alerting strategies.

## Exercise 1: CI/CD/CT Distinctions (Conceptual)

**Difficulty:** ⭐ Easy

For each scenario below, identify whether it describes **CI** (Continuous Integration), **CD** (Continuous Delivery/Deployment), or **CT** (Continuous Training):

1. Every Monday at 2 AM, the system pulls fresh data from the warehouse, retrains the model, and registers it as a candidate if all tests pass.

2. A developer opens a PR that changes the feature engineering pipeline. Automated tests run to verify the code compiles and unit tests pass.

3. After all tests pass on the main branch, the packaged model artifact is automatically pushed to the staging environment.

4. A PSI alert fires because input drift exceeded 0.25. The system automatically triggers a retraining job.

5. A GitHub Actions workflow runs `pytest`, `validate_data.py`, and `evaluate.py` on every pull request.

**Task:** Write your answers and explain the key characteristic that helped you decide.

## Exercise 2: Workflow Simulator (Coding)

**Difficulty:** ⭐⭐ Medium

Create an enhanced workflow simulator that:
- Accepts a list of steps with names, pass/fail status, and optional execution time in seconds
- Stops at the first failure
- Returns a structured report showing total time, success status, and which step failed (if any)

**Requirements:**
- Function signature: `run_enhanced_workflow(steps: list[dict]) -> dict`
- Each step dict has: `{"name": str, "passed": bool, "time_sec": float}`
- Return: `{"success": bool, "total_time": float, "steps_run": int, "failed_step": str | None, "log": list[str]}`

**Test with these pipelines:**
```python
pipeline_a = [
    {"name": "checkout", "passed": True, "time_sec": 2.3},
    {"name": "install deps", "passed": True, "time_sec": 45.1},
    {"name": "pytest", "passed": True, "time_sec": 12.8},
    {"name": "validate data", "passed": False, "time_sec": 3.2},
    {"name": "evaluate model", "passed": True, "time_sec": 8.5},
]

pipeline_b = [
    {"name": "checkout", "passed": True, "time_sec": 2.1},
    {"name": "install deps", "passed": True, "time_sec": 43.7},
    {"name": "pytest", "passed": True, "time_sec": 15.3},
    {"name": "validate data", "passed": True, "time_sec": 2.9},
    {"name": "evaluate model", "passed": True, "time_sec": 9.1},
]
```

In [ ]:
# Your solution here


## Exercise 3: Multi-Signal Release Gate (Coding)

**Difficulty:** ⭐⭐ Medium

Build a comprehensive release gate that checks multiple signals and provides detailed blocking reasons.

**Requirements:**
- Function signature: `release_gate(tests_pass: bool, data_valid: bool, accuracy: float, latency_p95: float, min_accuracy: float = 0.88, max_latency: float = 200.0) -> tuple[bool, list[str]]`
- Check all conditions:
  - Unit/contract tests must pass
  - Data validation must pass
  - Accuracy must meet or exceed threshold
  - P95 latency must be under max threshold
- Return `(allowed, reasons)` where `allowed` is `True` only if ALL checks pass
- Collect ALL failure reasons, not just the first one

**Test with these candidates:**
```python
candidates = [
    {"name": "v1.2-baseline", "tests_pass": True, "data_valid": True, 
     "accuracy": 0.91, "latency_p95": 145.0},
    {"name": "v1.3-fast", "tests_pass": True, "data_valid": True, 
     "accuracy": 0.85, "latency_p95": 89.0},
    {"name": "v1.3-accurate", "tests_pass": False, "data_valid": True, 
     "accuracy": 0.94, "latency_p95": 156.0},
    {"name": "v1.4-broken", "tests_pass": False, "data_valid": False, 
     "accuracy": 0.82, "latency_p95": 312.0},
]
```

In [ ]:
# Your solution here


## Exercise 4: Rollout Pattern Selection (Conceptual)

**Difficulty:** ⭐⭐ Medium

For each scenario, choose the most appropriate rollout pattern (**blue-green**, **canary**, or **shadow**) and explain why:

1. You're replacing a critical fraud detection model. The new model uses a completely different architecture (tree-based → neural network). You want to measure real-world performance without affecting any transactions.

2. You need to deploy a model update before 9 AM Monday for a marketing campaign. If anything goes wrong, you must be able to switch back instantly.

3. You're deploying a new recommendation model. You want to verify it performs well in production but want to minimize risk. Your infrastructure budget is tight.

4. You're testing a radically different NLP model that might have unexpected failure modes. You want real user data to flow through it, but you're not ready to show users the results yet.

**Task:** Write your answers with justification for each choice.

## Exercise 5: Canary Decision Engine (Coding)

**Difficulty:** ⭐⭐ Medium

Implement a canary promotion decision system that follows a pre-agreed rule.

**Requirements:**
- Function signature: `canary_decision(stable_errors: np.ndarray, canary_errors: np.ndarray, is_canary: np.ndarray, max_ratio: float = 1.15, min_requests: int = 100) -> dict`
- Calculate observed error rates for both stable and canary traffic
- Decision rule: PROMOTE if `canary_error_rate / stable_error_rate <= max_ratio` AND `canary_requests >= min_requests`
- Return: `{"decision": str, "canary_rate": float, "stable_rate": float, "ratio": float, "canary_requests": int, "reason": str}`

**Test with simulated traffic:**
```python
import numpy as np
rng = np.random.default_rng(42)

# Scenario 1: Good canary, enough samples
N = 2000
stable_err_1 = rng.random(N) < 0.035
canary_err_1 = rng.random(N) < 0.038
is_canary_1 = rng.random(N) < 0.10

# Scenario 2: Bad canary
stable_err_2 = rng.random(N) < 0.035
canary_err_2 = rng.random(N) < 0.055
is_canary_2 = rng.random(N) < 0.10

# Scenario 3: Good canary but too few samples
stable_err_3 = rng.random(N) < 0.035
canary_err_3 = rng.random(N) < 0.037
is_canary_3 = rng.random(N) < 0.02  # only 2%
```

In [ ]:
import numpy as np

# Your solution here


## Exercise 6: PSI Calculation from Scratch (Coding)

**Difficulty:** ⭐⭐⭐ Hard

Implement Population Stability Index calculation with proper handling of edge cases.

**Requirements:**
- Function signature: `calculate_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10, epsilon: float = 1e-6) -> tuple[float, pd.DataFrame]`
- Bin edges must be calculated from the `expected` (training) distribution
- Handle outliers by setting first bin edge to `-inf` and last to `+inf`
- Apply epsilon to prevent log(0) errors
- Return both the PSI score and a DataFrame showing per-bin calculations
- Add interpretation: return a tuple `(psi_score, details_df, interpretation: str)` where interpretation is "stable", "moderate shift", or "major shift"

**Test cases:**
```python
rng = np.random.default_rng(123)

# Training data: average session duration in minutes
training = rng.normal(45, 12, 10000)

# Test scenarios
stable = rng.normal(45, 12, 2000)
moderate = rng.normal(48, 14, 2000)
major = rng.normal(60, 20, 2000)
```

**Expected DataFrame columns:** `bin_id`, `expected_pct`, `actual_pct`, `psi_contribution`

In [ ]:
import numpy as np
import pandas as pd

# Your solution here


## Exercise 7: Multi-Feature PSI Monitor (Coding)

**Difficulty:** ⭐⭐⭐ Hard

Build a monitoring system that tracks PSI across multiple features and prioritizes alerts.

**Requirements:**
- Function signature: `monitor_drift(training_data: pd.DataFrame, live_data: pd.DataFrame, feature_importance: dict[str, float]) -> pd.DataFrame`
- Calculate PSI for each numeric column
- Weight the severity by feature importance (high importance features with high PSI are top priority)
- Return a DataFrame sorted by `alert_priority = psi * importance`, with columns: `feature`, `psi`, `importance`, `alert_priority`, `status`
- Status should be "ok", "watch", or "alert" based on PSI thresholds

**Test with this dataset:**
```python
rng = np.random.default_rng(456)

# Training data
train_df = pd.DataFrame({
    'age': rng.normal(35, 10, 5000),
    'income': rng.normal(50000, 15000, 5000),
    'credit_score': rng.normal(680, 80, 5000),
    'loan_amount': rng.normal(25000, 10000, 5000),
})

# Live data with varying drift
live_df = pd.DataFrame({
    'age': rng.normal(35.5, 10.5, 1000),        # stable
    'income': rng.normal(53000, 18000, 1000),   # moderate shift
    'credit_score': rng.normal(650, 90, 1000),  # major shift
    'loan_amount': rng.normal(26000, 11000, 1000),  # stable
})

importance = {
    'credit_score': 0.45,
    'income': 0.30,
    'loan_amount': 0.15,
    'age': 0.10,
}
```

In [ ]:
import numpy as np
import pandas as pd

# Your solution here


## Exercise 8: Alert Routing System (Coding)

**Difficulty:** ⭐⭐ Medium

Design an alert routing system that triages monitoring signals to appropriate channels.

**Requirements:**
- Function signature: `route_alert(metric_name: str, value: float, thresholds: dict) -> dict`
- Routes: `"page"`, `"ticket"`, `"digest"`, or `"silent"`
- Return: `{"route": str, "priority": str, "message": str, "actionable": bool}`
- Priority levels: "P0" (page now), "P1" (ticket), "P2" (digest), "P3" (silent)

**Routing rules:**
```python
thresholds = {
    'error_rate': {'page': 0.05, 'ticket': 0.02, 'type': 'upper'},
    'psi_top_feature': {'page': 0.25, 'ticket': 0.10, 'type': 'upper'},
    'latency_p95': {'page': 500, 'ticket': 300, 'type': 'upper'},
    'data_freshness_hours': {'page': 4, 'ticket': 2, 'type': 'upper'},
    'daily_predictions': {'page': 1000, 'ticket': 5000, 'type': 'lower'},
}
```

**Test with these events:**
```python
events = [
    ('error_rate', 0.08),
    ('psi_top_feature', 0.15),
    ('latency_p95', 450),
    ('data_freshness_hours', 5.5),
    ('daily_predictions', 800),
    ('error_rate', 0.015),
]
```

In [ ]:
# Your solution here


## Exercise 9: Alert Fatigue Analysis (Coding)

**Difficulty:** ⭐⭐⭐ Hard

Analyze a month of monitoring data to identify alert fatigue patterns and recommend threshold adjustments.

**Requirements:**
- Function signature: `analyze_alert_fatigue(alerts_df: pd.DataFrame) -> dict`
- Input DataFrame has columns: `timestamp`, `metric`, `value`, `route`, `acknowledged`, `actionable`
- Calculate:
  - Total pages sent
  - Page acknowledgment rate
  - Percentage of pages marked actionable
  - Metrics with highest false positive rate (pages sent but not actionable)
  - Recommended threshold adjustments
- Return comprehensive report dict

**Test data:**
```python
import pandas as pd
rng = np.random.default_rng(789)

# Simulate 30 days of alerts
dates = pd.date_range('2026-07-01', '2026-07-30', freq='H')
n = len(dates)

alerts_df = pd.DataFrame({
    'timestamp': dates,
    'metric': rng.choice(['error_rate', 'psi', 'latency', 'freshness'], n),
    'value': rng.uniform(0, 1, n),
    'route': rng.choice(['page', 'ticket', 'digest'], n, p=[0.15, 0.35, 0.50]),
    'acknowledged': rng.choice([True, False], n, p=[0.7, 0.3]),
    'actionable': rng.choice([True, False], n, p=[0.6, 0.4]),
})
```

**Expected output keys:** `total_pages`, `ack_rate`, `actionable_rate`, `worst_offenders`, `recommendations`

In [ ]:
import numpy as np
import pandas as pd

# Your solution here


## Exercise 10: End-to-End Gate Simulation (Challenge)

**Difficulty:** ⭐⭐⭐ Hard

Build a complete deployment simulation that integrates CI gates, canary testing, and drift monitoring.

**Requirements:**
- Simulate a model deployment pipeline with multiple stages
- Stage 1: CI gates (tests, data validation, accuracy threshold)
- Stage 2: Canary deployment (5% traffic, monitor for 1000 requests)
- Stage 3: PSI check (compare canary input distribution to training)
- If all stages pass, promote to 100%; otherwise rollback

**Function signature:**
```python
def deploy_pipeline(
    tests_pass: bool,
    data_valid: bool,
    accuracy: float,
    training_dist: np.ndarray,
    stable_error_rate: float,
    canary_error_rate: float,
    canary_input_dist: np.ndarray,
    min_accuracy: float = 0.88,
    max_error_ratio: float = 1.15,
    max_psi: float = 0.25
) -> dict
```

**Return:** `{"stage": str, "decision": str, "details": dict, "final_status": str}`

**Test with three scenarios:**
1. Perfect deployment: all gates pass
2. Failed CI: accuracy too low
3. Failed canary: PSI too high (distribution shifted)

In [ ]:
import numpy as np

# Your solution here


## Exercise 11: CT Trigger Logic (Challenge)

**Difficulty:** ⭐⭐⭐ Hard

Design a Continuous Training trigger system that decides when to retrain based on multiple signals.

**Requirements:**
- Function signature: `should_retrain(days_since_train: int, psi_scores: dict[str, float], performance_drop: float, feature_importance: dict[str, float], schedule_day: str, force: bool = False) -> tuple[bool, str]`
- Trigger retraining if:
  - `force=True` (manual override)
  - Scheduled day (e.g., "Monday")
  - Any high-importance feature (>0.3) has PSI > 0.25
  - Performance dropped by more than 5% AND any feature PSI > 0.15
  - Days since training > 30 AND weighted average PSI > 0.15
- Return: `(should_retrain: bool, reason: str)`

**Test scenarios:**
```python
feature_importance = {
    'credit_score': 0.45,
    'income': 0.30,
    'debt_ratio': 0.15,
    'age': 0.10,
}

scenarios = [
    # (days, psi_scores, perf_drop, current_day, force)
    (7, {'credit_score': 0.05, 'income': 0.08, 'debt_ratio': 0.03, 'age': 0.04}, 0.01, 'Monday', False),
    (15, {'credit_score': 0.28, 'income': 0.12, 'debt_ratio': 0.09, 'age': 0.05}, 0.02, 'Wednesday', False),
    (12, {'credit_score': 0.18, 'income': 0.16, 'debt_ratio': 0.11, 'age': 0.08}, 0.07, 'Friday', False),
    (35, {'credit_score': 0.14, 'income': 0.17, 'debt_ratio': 0.13, 'age': 0.19}, 0.03, 'Tuesday', False),
    (3, {'credit_score': 0.02, 'income': 0.03, 'debt_ratio': 0.01, 'age': 0.02}, 0.00, 'Thursday', True),
]
```

In [ ]:
# Your solution here


## Exercise 12: Monitoring Dashboard Simulator (Challenge)

**Difficulty:** ⭐⭐⭐ Hard

Create a comprehensive monitoring summary that executives and engineers can both understand.

**Requirements:**
- Function signature: `generate_monitoring_report(metrics_df: pd.DataFrame, audience: str) -> str`
- Input: DataFrame with columns `timestamp`, `requests`, `errors`, `latency_p95`, `latency_p99`, `accuracy`, `psi_max`
- `audience` can be `"executive"` or `"engineer"`
- Executive view: high-level KPIs, trends, red flags only
- Engineer view: detailed metrics, percentiles, drift scores, actionable items

**Test data:**
```python
rng = np.random.default_rng(999)
dates = pd.date_range('2026-08-01', '2026-08-26', freq='D')

metrics_df = pd.DataFrame({
    'timestamp': dates,
    'requests': rng.integers(8000, 12000, len(dates)),
    'errors': rng.integers(50, 200, len(dates)),
    'latency_p95': rng.uniform(120, 180, len(dates)),
    'latency_p99': rng.uniform(200, 300, len(dates)),
    'accuracy': rng.uniform(0.88, 0.93, len(dates)),
    'psi_max': rng.uniform(0.02, 0.30, len(dates)),
})
```

Return formatted string reports appropriate for each audience.

In [ ]:
import numpy as np
import pandas as pd

# Your solution here


---

## 🎓 Reflection Questions

After completing these exercises, consider:

1. Why is it critical that PSI bins are calculated from training data and frozen, never recalculated on live data?

2. What are the tradeoffs between blue-green, canary, and shadow deployments? When would you choose each?

3. How does alert routing prevent fatigue? Why is "actionable" the key criterion?

4. What's the relationship between CI/CD/CT? How does CT differ from traditional DevOps automation?

5. Why must canary decision rules be pre-agreed before looking at the data?

---

**Next:** Check your solutions against `solutions.ipynb` and review the best practices notes.